# Расчёт рекомендаций (часть 1, этапы 1–3)

Ноутбук считает рекомендации для интернет-магазина по данным Retailrocket:
разбивает события по времени, строит базовую модель (топ популярных),
обучает ALS, подбирает его гиперпараметры Optuna и рассчитывает похожие
товары. Все эксперименты логируются в MLflow, артефакты выгружаются в S3.

Вся логика вынесена в пакет `recsys` (`recsys/data.py`, `recsys/features.py`,
`recsys/models.py`, `recsys/metrics.py`) — те же функции используют стадии DVC
и шаги DAG, поэтому в ноутбуке остаются только вызовы и выводы.

## Инициализация

Импорты, параметры пайплайна из `params.yaml` и подключение к MLflow.
Значения параметров не задаются в ноутбуке руками: даты сплитов, веса событий
и гиперпараметры лежат в `params.yaml` в корне репозитория.

In [1]:
import json
import os
import sys
import warnings

import mlflow
import numpy as np
import optuna
import pandas as pd
import yaml

# ноутбук лежит в notebooks/, пакет recsys — в корне репозитория
sys.path.insert(0, os.path.abspath(".."))

from recsys.config import (
    ALS_MODEL_PATH,
    ID_MAPS_PATH,
    RECS_DEFAULT_PATH,
    S3_BUCKET,
    S3_PREFIX,
    SEED,
    SIMILAR_PATH,
)
from recsys.data import load_events, split_events
from recsys.features import (
    build_matrix,
    decode,
    encode,
    positives,
    user_history_purchases,
    weight_events,
)
from recsys.metrics import evaluate
from recsys.models import als_recommend, fit_als, save_als, similar_items, top_popular
from recsys.s3_io import list_objects, upload_file_if_changed, write_parquet

# в выводе не нужны предупреждения библиотек и ссылки на запуски, которые
# MLflow 3.x печатает сам: идентификаторы запусков печатаются явно ниже
warnings.filterwarnings("ignore")
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 50)

# --- Параметры проекта ---
with open("../params.yaml") as fd:
    params = yaml.safe_load(fd)

SPLIT_DATES   = {k: v for k, v in params.items() if k.startswith("split_")}
EVENT_WEIGHTS = params["event_weights"]        # view 1, addtocart 5, transaction 10
ALS_FACTORS   = params["als"]["factors"]
ALS_REG       = params["als"]["regularization"]
ALS_ITERS     = params["als"]["iterations"]
N_TRIALS      = params["optuna"]["n_trials"]   # число попыток подбора Optuna
TOP_K         = params["top_k"]                # глубина выдачи, на ней считаем метрики
N_CANDIDATES  = params["n_candidates"]         # длина списка популярного
N_SIMILAR     = params["n_similar"]            # сколько похожих товаров на товар
MODELS_DIR    = "../models"
DATA_DIR      = "../data"

EXPERIMENT_NAME = "final_project_recsys"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
EXPERIMENT_ID = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

print("параметры сплита:", SPLIT_DATES)
print("веса событий:", EVENT_WEIGHTS)
print("эксперимент MLflow:", EXPERIMENT_NAME, "| id:", EXPERIMENT_ID)

/home/user/venvs/mle-pr-final/lib/python3.12/site-packages/pydantic/_internal/_fields.py:149: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


/home/user/venvs/mle-pr-final/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


параметры сплита: {'split_train_fit': '2015-05-03', 'split_valid': '2015-08-04', 'split_labels': '2015-08-19', 'split_test': '2015-09-03', 'split_end': '2015-09-19'}
веса событий: {'view': 1, 'addtocart': 5, 'transaction': 10}
эксперимент MLflow: final_project_recsys | id: 9


## Этап 1. Данные и разбиение по времени

Читаем все события из таблицы `retailrocket_events` в Postgres. Это
единственное обращение к базе — дальше всё считается в памяти.

In [2]:
events = load_events()

print(f"событий: {len(events):,}")
print(f"период: {events['event_ts'].min()} — {events['event_ts'].max()}")
print(f"визитёров: {events['visitorid'].nunique():,}")
print(f"товаров: {events['itemid'].nunique():,}")
print(events["event"].value_counts().to_dict())

событий: 2,756,101
период: 2015-05-03 03:00:04.384000 — 2015-09-18 02:59:47.788000
визитёров: 1,407,580
товаров: 235,061


{'view': 2664312, 'addtocart': 69332, 'transaction': 22457}


Разбиение делаем по времени, а не случайно: рекомендательная система
предсказывает будущее поведение, и случайный сплит дал бы заглядывание вперёд.
Границы окон — в `params.yaml`, верхняя граница везде строгая, поэтому событие
попадает ровно в одно окно.

- `train_fit` — обучение ALS при подборе гиперпараметров;
- `valid` — оценка recall@10 в Optuna, тест при подборе не используется;
- `train` = `train_fit` + `valid` — обучение финальной модели;
- `labels` — источник таргета для ранжировщика (этап 4);
- `test` — единственное окно итоговых метрик.

In [3]:
parts = split_events(events, SPLIT_DATES)

split_info = pd.DataFrame(
    [
        {
            "окно": name,
            "начало": part["event_ts"].min().date(),
            "конец": part["event_ts"].max().date(),
            "событий": len(part),
            "визитёров": part["visitorid"].nunique(),
            "товаров": part["itemid"].nunique(),
        }
        for name, part in parts.items()
    ]
)
display(split_info)

# окна не пересекаются: сумма четырёх непересекающихся окон равна всему объёму
n_windows = sum(len(parts[name]) for name in ["train_fit", "valid", "labels", "test"])
print(f"сумма окон: {n_windows:,} | всего событий: {len(events):,}")

,окно,начало,конец,событий,визитёров,товаров
0,train_fit,2015-05-03,2015-08-03,1950977,990109,201952
1,valid,2015-08-04,2015-08-18,269662,155878,77071
2,labels,2015-08-19,2015-09-02,267469,152373,76354
3,test,2015-09-03,2015-09-18,267993,154021,76136
4,train,2015-05-03,2015-08-18,2220639,1131714,213612


сумма окон: 2,756,101 | всего событий: 2,756,101


Веса событий: `view` — 1, `addtocart` — 5, `transaction` — 10. Сырой вес
пары «визитёр — товар» равен сумме весов её событий, а в матрицу кладём
`log1p` от этой суммы: медиана событий на визитёра равна 1, но максимум
доходит до тысяч, логарифм гасит вклад накрутчиков.

Матрицу строим дважды — на `train_fit` для подбора гиперпараметров и на
`train` для финальной модели, чтобы при подборе модель не видела валидацию.

In [4]:
pairs_fit = weight_events(parts["train_fit"], EVENT_WEIGHTS)
matrix_fit, id_maps_fit = build_matrix(pairs_fit)

pairs_train = weight_events(parts["train"], EVENT_WEIGHTS)
matrix_train, id_maps_train = build_matrix(pairs_train)

print(f"train_fit: матрица {matrix_fit.shape}, ненулевых {matrix_fit.nnz:,}")
print(f"train:     матрица {matrix_train.shape}, ненулевых {matrix_train.nnz:,}")
print(f"плотность train: {matrix_train.nnz / np.prod(matrix_train.shape):.2e}")
display(pairs_train[["weight", "value"]].describe().round(3))

train_fit: матрица (990109, 201952), ненулевых 1,513,542
train:     матрица (1131714, 213612), ненулевых 1,725,395
плотность train: 7.14e-06


,weight,value
count,1725395.000,1725395.000
mean,1.510,0.813
std,2.296,0.350
min,1.000,0.693
25%,1.000,0.693
50%,1.000,0.693
75%,1.000,0.693
max,308.000,5.733


Позитивы для оценки — пары «визитёр — товар», по которым в окне оценки
было добавление в корзину или покупка. Просмотры позитивами не считаются:
целевое действие кейса — `addtocart`.

База усреднения одна и та же для всех моделей: пользователи, у которых есть
хотя бы один позитив в окне оценки **и** хотя бы одно событие в истории, на
которой обучалась модель. Пользователя без истории модель в принципе не может
ранжировать, поэтому включать его в среднее бессмысленно.

In [5]:
pos_valid = positives(parts["valid"])
pos_test = positives(parts["test"])

# в базе усреднения — только пользователи с историей в окне обучения
base_valid = np.intersect1d(
    pos_valid["visitorid"].unique(), parts["train_fit"]["visitorid"].unique()
)
base_test = np.intersect1d(
    pos_test["visitorid"].unique(), parts["train"]["visitorid"].unique()
)

print(f"valid: позитивных пар {len(pos_valid):,}, "
      f"пользователей с позитивом {pos_valid['visitorid'].nunique():,}, "
      f"в базе усреднения {len(base_valid):,}")
print(f"test:  позитивных пар {len(pos_test):,}, "
      f"пользователей с позитивом {pos_test['visitorid'].nunique():,}, "
      f"в базе усреднения {len(base_test):,}")

valid: позитивных пар 6,703, пользователей с позитивом 4,158, в базе усреднения 576
test:  позитивных пар 6,448, пользователей с позитивом 3,973, в базе усреднения 383


Дальше — служебные функции ноутбука: популярность товара для `novelty`,
рекомендации ALS в сырых идентификаторах, оценка модели сразу на двух окнах и
запись метрик в MLflow. Метрики на валидации всегда считает модель, обученная
на `train_fit`, метрики на тесте — модель с теми же параметрами, обученная на
`train`.

In [6]:
def item_popularity(events_part):
    """доля визитёров окна, взаимодействовавших с товаром (для novelty)"""
    return (
        events_part.groupby("itemid")["visitorid"].nunique()
        / events_part["visitorid"].nunique()
    )


def bought_enc(events_part, id_maps):
    """пары (user_enc, item_enc) с покупкой в истории — их не рекомендуем"""
    bought = user_history_purchases(events_part)
    enc = pd.DataFrame({
        "user_enc": encode(id_maps, "user", bought["visitorid"]),
        "item_enc": encode(id_maps, "item", bought["itemid"]),
    })
    return enc[(enc["user_enc"] >= 0) & (enc["item_enc"] >= 0)]


def als_recs(model, matrix, id_maps, users, bought, k):
    """рекомендации ALS для списка сырых visitorid, в выдаче — сырые id"""
    enc_users = encode(id_maps, "user", users)
    recs = als_recommend(model, matrix, enc_users, n=k, exclude=bought)
    return pd.DataFrame({
        "visitorid": decode(id_maps, "user", recs["user_enc"]),
        "itemid": decode(id_maps, "item", recs["item_enc"]),
        "score": recs["score"].to_numpy(),
        "rank": recs["rank"].to_numpy(),
    })


def score_model(recs_valid, recs_test):
    """метрики модели на валидации и на тесте с общей базой усреднения"""
    m_valid = evaluate(recs_valid, pos_valid, TOP_K, catalog_fit, pop_fit, base_valid)
    m_test = evaluate(recs_test, pos_test, TOP_K, catalog_train, pop_train, base_test)
    return m_valid, m_test


def log_metrics(metrics, prefix):
    """кладёт словарь метрик в MLflow с префиксом окна оценки"""
    named = {f"{prefix}_{name}_{TOP_K}": value for name, value in metrics.items()}
    mlflow.log_metrics(named)


def metrics_frame(m_valid, m_test):
    """таблица метрик модели по двум окнам"""
    return pd.DataFrame([m_valid, m_test], index=["valid", "test"]).round(4)


pop_fit = item_popularity(parts["train_fit"])
pop_train = item_popularity(parts["train"])
catalog_fit, catalog_train = matrix_fit.shape[1], matrix_train.shape[1]
bought_fit = bought_enc(parts["train_fit"], id_maps_fit)
bought_train = bought_enc(parts["train"], id_maps_train)

print(f"каталог train_fit: {catalog_fit:,} | каталог train: {catalog_train:,}")
print(f"купленных пар: train_fit {len(bought_fit):,} | train {len(bought_train):,}")

каталог train_fit: 201,952 | каталог train: 213,612
купленных пар: train_fit 14,991 | train 17,092


**Выводы по разбиению.**

- **Объём.** Все события помещаются в память, выгрузка по частям не нужна.
- **Окна.** Три окна оценки по 15 дней сопоставимы по объёму, поэтому метрики на
  валидации и на тесте сравнимы между собой.
- **Разреженность.** Матрица «пользователи — товары» крайне разрежена, что
  типично для e-commerce: подавляющее большинство визитёров посмотрели
  единицы товаров, в среднем на визитёра приходится полтора товара.
- **База усреднения.** Пользователей с целевым действием в окне оценки немного:
  добавления в корзину совершают около 3 % визитёров, и лишь у малой их части
  есть история в окне обучения. Это ограничивает абсолютные значения метрик и
  делает их шумными, но база одинакова для всех моделей.

## Этап 2. Базовая модель: топ популярных

Рассчитаем рекомендации как топ популярных. Популярность считаем по числу
визитёров, добавивших товар в корзину, — это целевое действие кейса, а не
просто просмотры. Список один и тот же для всех пользователей, из него
убираются товары, которые пользователь уже купил.

In [7]:
top_pop_fit = top_popular(parts["train_fit"], k=N_CANDIDATES)
top_pop_train = top_popular(parts["train"], k=N_CANDIDATES)
display(top_pop_train.head(10))


def popular_recs(top_pop, users, events_part, k):
    """один список популярного каждому пользователю, минус его покупки"""
    items = top_pop["itemid"].to_numpy()
    recs = pd.DataFrame({
        "visitorid": np.repeat(users, len(items)),
        "itemid": np.tile(items, len(users)),
    })
    bought = user_history_purchases(events_part).assign(bought=1)
    recs = recs.merge(bought, on=["visitorid", "itemid"], how="left")
    recs = recs[recs["bought"].isna()].drop(columns="bought")
    recs["rank"] = recs.groupby("visitorid").cumcount() + 1
    return recs[recs["rank"] <= k].reset_index(drop=True)


recs_pop_valid = popular_recs(top_pop_fit, base_valid, parts["train_fit"], TOP_K)
recs_pop_test = popular_recs(top_pop_train, base_test, parts["train"], TOP_K)
pop_valid_metrics, pop_test_metrics = score_model(recs_pop_valid, recs_pop_test)
display(metrics_frame(pop_valid_metrics, pop_test_metrics))

,itemid,score,rank
0,461686,0.000170,1
1,312728,0.000102,2
2,409804,0.000097,3
3,29196,0.000086,4
4,48030,0.000075,5
5,257040,0.000070,6
6,445351,0.000065,7
7,7943,0.000065,8
8,441852,0.000063,9
9,316753,0.000062,10


,precision,recall,map,ndcg,coverage,novelty
valid,0.0040,0.0203,0.0171,0.0204,0.0001,10.9087
test,0.0021,0.0070,0.0050,0.0075,0.0001,10.8550


Логируем базовую модель в MLflow отдельным запуском.

In [8]:
with mlflow.start_run(run_name="1_base_model") as run:
    mlflow.log_params({
        "model": "top_popular",
        "popularity_by": "addtocart",
        "top_k": TOP_K,
        "n_candidates": N_CANDIDATES,
        **SPLIT_DATES,
    })
    mlflow.log_metrics({"users_valid": len(base_valid), "users_test": len(base_test)})
    log_metrics(pop_valid_metrics, "valid")
    log_metrics(pop_test_metrics, "test")

    top_popular_path = f"{DATA_DIR}/top_popular.parquet"
    top_pop_train.to_parquet(top_popular_path, index=False)
    write_parquet(top_pop_train, RECS_DEFAULT_PATH)
    mlflow.log_artifact(top_popular_path)
    mlflow.log_artifact("../params.yaml")
    run_id_base = run.info.run_id

print("run 1_base_model:", run_id_base)

run 1_base_model: d6d275f6660448a0bde89ef7ff136201


**Вывод по базовой модели.** Топ популярных даёт ненулевые, но очень
низкие precision и recall и почти нулевой coverage: всем пользователям
показывается один и тот же десяток товаров. Это нижняя планка, которую обязана
превзойти персональная модель.

## Этап 3. Персональные рекомендации (ALS)

ALS из `implicit` раскладывает матрицу «пользователи — товары» на
скрытые факторы и выдаёт персональный скор для каждой пары. Стартовые
гиперпараметры берём из `params.yaml` — это значения, проверенные в четвёртом
спринте: 64 фактора, регуляризация 0.05, 20 итераций.

Уже купленные товары из выдачи убираются, а просмотренные и отложенные —
нет: пользователь мог вернуться к товару, который смотрел, и именно это
событие мы предсказываем.

In [9]:
als_fit = fit_als(matrix_fit, ALS_FACTORS, ALS_REG, ALS_ITERS, SEED)
als_train = fit_als(matrix_train, ALS_FACTORS, ALS_REG, ALS_ITERS, SEED)

recs_als_valid = als_recs(
    als_fit, matrix_fit, id_maps_fit, base_valid, bought_fit, TOP_K
)
recs_als_test = als_recs(
    als_train, matrix_train, id_maps_train, base_test, bought_train, TOP_K
)
als_valid_metrics, als_test_metrics = score_model(recs_als_valid, recs_als_test)

display(recs_als_test.head())
display(metrics_frame(als_valid_metrics, als_test_metrics))

,visitorid,itemid,score,rank
0,155,62549,0.002146,1
1,155,299222,0.002129,2
2,155,273936,0.001916,3
3,155,200425,0.001914,4
4,155,17108,0.001896,5


,precision,recall,map,ndcg,coverage,novelty
valid,0.0104,0.0603,0.0254,0.0374,0.0033,11.7043
test,0.0060,0.0235,0.0070,0.0141,0.0025,11.6955


In [10]:
with mlflow.start_run(run_name="Stage_3_ALS") as run:
    mlflow.log_params({
        "model": "als",
        "factors": ALS_FACTORS,
        "regularization": ALS_REG,
        "iterations": ALS_ITERS,
        "seed": SEED,
        "top_k": TOP_K,
        **SPLIT_DATES,
    })
    mlflow.log_metrics({"users_valid": len(base_valid), "users_test": len(base_test)})
    log_metrics(als_valid_metrics, "valid")
    log_metrics(als_test_metrics, "test")
    mlflow.log_artifact("../params.yaml")
    run_id_als = run.info.run_id

print("run Stage_3_ALS:", run_id_als)

run Stage_3_ALS: 7560ad7b2d4744acbb67a6962c35703a


**Вывод по ALS со стартовыми параметрами.** Персонализация даёт кратный
прирост по precision и recall относительно топа популярных и на порядки
больший coverage: модель достаёт из каталога тысячи товаров вместо десяти.
Novelty тоже выше — рекомендуются не только самые заезженные позиции.

### Подбор гиперпараметров (Optuna)

Подбираем число факторов, регуляризацию и число итераций по recall@10 на
валидации. Обучение внутри подбора идёт только на `train_fit`, тестовое окно
не используется вообще. Каждая попытка логируется вложенным запуском MLflow.

In [11]:
def objective(trial):
    """recall@10 на валидации; модель обучается только на train_fit"""
    factors = trial.suggest_categorical("factors", [32, 64, 128])
    regularization = trial.suggest_float("regularization", 1e-3, 1e-1, log=True)
    iterations = trial.suggest_categorical("iterations", [10, 15, 20])

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        model = fit_als(matrix_fit, factors, regularization, iterations, SEED)
        recs = als_recs(model, matrix_fit, id_maps_fit, base_valid, bought_fit, TOP_K)
        recall = evaluate(recs, pos_valid, TOP_K, users=base_valid)["recall"]
        mlflow.log_params({
            "factors": factors,
            "regularization": regularization,
            "iterations": iterations,
        })
        mlflow.log_metric(f"valid_recall_{TOP_K}", recall)
    return recall


study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
with mlflow.start_run(run_name="Stage_3_ALS_Optuna") as run:
    study.optimize(objective, n_trials=N_TRIALS)
    best_named = {f"best_{name}": value for name, value in study.best_params.items()}
    mlflow.log_params(best_named)
    mlflow.log_metric(f"best_valid_recall_{TOP_K}", study.best_value)
    run_id_optuna = run.info.run_id

print("лучшие параметры:", study.best_params)
print(f"лучший recall@{TOP_K} на валидации: {study.best_value:.4f}")
print("run Stage_3_ALS_Optuna:", run_id_optuna)

лучшие параметры: {'factors': 128, 'regularization': 0.008168455894760165, 'iterations': 10}
лучший recall@10 на валидации: 0.0816
run Stage_3_ALS_Optuna: bb3da50d62a94d3a99e3c6952fe3f24d


In [12]:
trials = study.trials_dataframe()[
    ["number", "params_factors", "params_regularization", "params_iterations", "value"]
]
display(trials.sort_values("value", ascending=False).round(4))

,number,params_factors,params_regularization,params_iterations,value
3,3,128,0.0082,10,0.0816
10,10,128,0.0053,15,0.0769
11,11,128,0.0056,15,0.0769
2,2,128,0.0112,20,0.0755
4,4,128,0.0022,20,0.0755
9,9,128,0.0019,20,0.0755
0,0,64,0.0158,10,0.0666
8,8,64,0.0045,20,0.0614
6,6,64,0.0211,20,0.0603
7,7,64,0.0757,20,0.0603


### Финальная модель ALS

Переобучаем ALS с лучшими параметрами: на `train_fit` — чтобы получить
полный набор метрик на валидации, и на `train` — это и есть финальная модель,
метрики которой считаются на тесте. Лучшие параметры записаны в `params.yaml`
в секцию `als_best`.

In [13]:
best = study.best_params
als_best_fit = fit_als(
    matrix_fit, best["factors"], best["regularization"], best["iterations"], SEED
)
als_best_train = fit_als(
    matrix_train, best["factors"], best["regularization"], best["iterations"], SEED
)

recs_best_valid = als_recs(
    als_best_fit, matrix_fit, id_maps_fit, base_valid, bought_fit, TOP_K
)
recs_best_test = als_recs(
    als_best_train, matrix_train, id_maps_train, base_test, bought_train, TOP_K
)
best_valid_metrics, best_test_metrics = score_model(recs_best_valid, recs_best_test)
display(metrics_frame(best_valid_metrics, best_test_metrics))

,precision,recall,map,ndcg,coverage,novelty
valid,0.0134,0.0816,0.0388,0.0536,0.0048,12.0481
test,0.0052,0.0257,0.0119,0.0180,0.0034,12.0094


Сохраняем модель и маппинг идентификаторов. Маппинг пишется parquet-ом, а
не pickle-ом: файл читают и ноутбук на Python 3.12, и шаги DAG в образе
Airflow на 3.11.

In [14]:
os.makedirs(MODELS_DIR, exist_ok=True)
als_path = f"{MODELS_DIR}/als_model.npz"
id_maps_path = f"{DATA_DIR}/id_maps.parquet"

save_als(als_best_train, als_path)
id_maps_train.to_parquet(id_maps_path, index=False)

print(f"размер файла модели: {os.path.getsize(als_path) / 1e6:.0f} МБ")
print(f"строк в маппинге идентификаторов: {len(id_maps_train):,}")

размер файла модели: 689 МБ
строк в маппинге идентификаторов: 1,345,326


In [15]:
with mlflow.start_run(run_name="Stage_3_ALS_final") as run:
    mlflow.log_params({
        "model": "als_optuna",
        "factors": best["factors"],
        "regularization": best["regularization"],
        "iterations": best["iterations"],
        "seed": SEED,
        "top_k": TOP_K,
        "n_trials": N_TRIALS,
        **SPLIT_DATES,
    })
    mlflow.log_metrics({
        "users_valid": len(base_valid),
        "users_test": len(base_test),
        "matrix_users": matrix_train.shape[0],
        "matrix_items": matrix_train.shape[1],
        "matrix_nnz": matrix_train.nnz,
    })
    log_metrics(best_valid_metrics, "valid")
    log_metrics(best_test_metrics, "test")

    # artifact store MLflow — тот же бакет, что и хранилище артефактов проекта,
    # поэтому вместо копий тяжёлых файлов логируем манифест с их ключами
    manifest = {
        "als_model": f"s3://{S3_BUCKET}/{ALS_MODEL_PATH}",
        "als_model_bytes": os.path.getsize(als_path),
        "id_maps": f"s3://{S3_BUCKET}/{ID_MAPS_PATH}",
        "top_popular": f"s3://{S3_BUCKET}/{RECS_DEFAULT_PATH}",
        "similar_items": f"s3://{S3_BUCKET}/{SIMILAR_PATH}",
    }
    manifest_path = f"{DATA_DIR}/artifacts_s3.json"
    with open(manifest_path, "w") as fd:
        json.dump(manifest, fd, indent=2)

    mlflow.log_params({"als_model_s3": manifest["als_model"]})
    mlflow.log_artifact(manifest_path)
    mlflow.log_artifact("../params.yaml")
    run_id_final = run.info.run_id

print("run Stage_3_ALS_final:", run_id_final)

run Stage_3_ALS_final: 885cd4f949854323af1c883612f600b8


Модель и маппинг кладём в S3 — оттуда их берут шаги DAG. В git файл
модели не коммитится (он в `.gitignore`), а в MLflow к запуску приложен
манифест с ключами: исходящий канал до бакета — около 0,1 МБ/с, и копировать
те же сотни мегабайт второй раз в тот же бакет смысла нет. Повторная выгрузка
пропускается, если в бакете уже лежит объект того же размера.

In [16]:
write_parquet(id_maps_train, ID_MAPS_PATH)
upload_file_if_changed(als_path, ALS_MODEL_PATH)

print("ключи в бакете:")
for key in list_objects(f"{S3_PREFIX}/models"):
    print(" ", key)

ключи в бакете:


  recsys_final/models/als_model.npz
  recsys_final/models/id_maps.parquet


### Похожие товары (i2i)

Похожие товары считаются по тем же факторам ALS: для каждого товара из
каталога `train` берём 11 ближайших и выбрасываем сам товар. Такая таблица
нужна сервису для карточки товара и для онлайн-рекомендаций по последнему
просмотру.

In [17]:
with mlflow.start_run(run_name="Stage_3_Similar_Items") as run:
    sim = similar_items(als_best_train, np.arange(catalog_train), n=N_SIMILAR)
    similar = pd.DataFrame({
        "itemid": decode(id_maps_train, "item", sim["item_enc"]),
        "similar_itemid": decode(id_maps_train, "item", sim["similar_enc"]),
        "score": sim["score"].to_numpy(),
        "rank": sim["rank"].to_numpy(),
    })

    similar_path = f"{DATA_DIR}/similar_items.parquet"
    similar.to_parquet(similar_path, index=False)
    write_parquet(similar, SIMILAR_PATH)

    # в MLflow кладём выборку из таблицы, полная версия лежит в S3
    sample_path = f"{DATA_DIR}/similar_items_sample.parquet"
    similar.head(1000).to_parquet(sample_path, index=False)

    mlflow.log_params({"n_similar": N_SIMILAR, "factors": best["factors"]})
    mlflow.log_metrics({
        "similar_rows": len(similar),
        "similar_items_covered": similar["itemid"].nunique(),
    })
    mlflow.log_artifact(sample_path)
    run_id_similar = run.info.run_id

print("run Stage_3_Similar_Items:", run_id_similar)
print(f"строк в таблице похожих: {len(similar):,}")
print(f"товаров с похожими: {similar['itemid'].nunique():,}")
display(similar.head(11))

run Stage_3_Similar_Items: 19427c52069e420f85b9af5340714784
строк в таблице похожих: 2,136,120
товаров с похожими: 213,612


,itemid,similar_itemid,score,rank
0,3,293704,0.992633,1
1,3,7740,0.990155,2
2,3,142739,0.989154,3
3,3,400566,0.988569,4
4,3,48479,0.988498,5
5,3,153199,0.988406,6
6,3,111491,0.988359,7
7,3,355955,0.988221,8
8,3,45561,0.987932,9
9,3,187785,0.987857,10


## Результаты

Сравним три модели на валидации и на тесте по одной и той же базе
усреднения. Метрики на валидации получены моделями, обученными на `train_fit`,
метрики на тесте — теми же моделями, переобученными на `train`.

История для рекомендаций на тесте — то же окно `train` (последнее
событие 18.08): окно `labels` на этом этапе не используется ни при обучении,
ни при инференсе, оно понадобится ранжировщику на этапе 4.

In [18]:
by_model = {
    "Топ популярных": (pop_valid_metrics, pop_test_metrics),
    "ALS (стартовые параметры)": (als_valid_metrics, als_test_metrics),
    "ALS + Optuna": (best_valid_metrics, best_test_metrics),
}
results = pd.concat(
    {name: metrics_frame(*pair) for name, pair in by_model.items()},
    names=["модель", "окно"],
)
results = results.rename(columns=lambda c: f"{c}@{TOP_K}")
display(results)

precision@10  recall@10  map@10  ndcg@10  \
модель                    окно                                              
Топ популярных            valid        0.0040     0.0203  0.0171   0.0204   
                          test         0.0021     0.0070  0.0050   0.0075   
ALS (стартовые параметры) valid        0.0104     0.0603  0.0254   0.0374   
                          test         0.0060     0.0235  0.0070   0.0141   
ALS + Optuna              valid        0.0134     0.0816  0.0388   0.0536   
                          test         0.0052     0.0257  0.0119   0.0180   

                                 coverage@10  novelty@10  
модель                    окно                            
Топ популярных            valid       0.0001     10.9087  
                          test        0.0001     10.8550  
ALS (стартовые параметры) valid       0.0033     11.7043  
                          test        0.0025     11.6955  
ALS + Optuna              valid       0.0048     12.0481  
                          test        0.0034     12.0094

Запуски эксперимента в MLflow, их имена и идентификаторы.

In [19]:
runs = mlflow.search_runs(experiment_ids=[EXPERIMENT_ID])
runs = runs.rename(columns={"tags.mlflow.runName": "run_name"})
main_runs = runs[~runs["run_name"].str.startswith("trial_")]

display(main_runs[["run_name", "run_id", "status"]].reset_index(drop=True))
print(f"вложенных запусков trial_*: {len(runs) - len(main_runs)}")

,run_name,run_id,status
0,Stage_3_Similar_Items,19427c52069e420f85b9af5340714784,FINISHED
1,Stage_3_ALS_final,885cd4f949854323af1c883612f600b8,FINISHED
2,Stage_3_ALS_Optuna,bb3da50d62a94d3a99e3c6952fe3f24d,FINISHED
3,Stage_3_ALS,7560ad7b2d4744acbb67a6962c35703a,FINISHED
4,1_base_model,d6d275f6660448a0bde89ef7ff136201,FINISHED


вложенных запусков trial_*: 12


## Выводы по части 1

1. **Данные и разбиение.** События Retailrocket разбиты по времени на четыре
   окна: `train_fit`, `valid`, `labels` и `test`. Случайный сплит не
   используется — он дал бы заглядывание в будущее. Границы вынесены в
   `params.yaml`, верхняя граница окна строгая, поэтому окна не пересекаются.
2. **Веса и матрица.** Вес пары «визитёр — товар» — `log1p` от суммы весов
   событий (`view` 1, `addtocart` 5, `transaction` 10). Маппинг сырых и
   закодированных идентификаторов сохранён parquet-таблицей
   `recsys_final/models/id_maps.parquet`, pickle в проекте не используется.
3. **Базовая модель.** Топ популярных по числу добавлений в корзину задаёт
   нижнюю планку качества и покрывает исчезающе малую долю каталога.
4. **ALS.** Персональная модель превосходит базовую по всем метрикам точности
   и радикально по coverage. Подбор гиперпараметров Optuna по recall@10 на
   валидации дал дополнительный прирост; тестовое окно при подборе не
   использовалось.
5. **Артефакты.** В S3 выгружены `models/als_model.npz`,
   `models/id_maps.parquet`, `recommendations/top_popular.parquet` и
   `recommendations/similar_items.parquet`. К запускам MLflow приложены
   `params.yaml`, топ популярных, выборка из таблицы похожих товаров и
   манифест с ключами тяжёлых артефактов. Файл модели в git не коммитится
   из-за размера.
6. **Ограничения.** Абсолютные значения precision и recall низкие, и это
   нормально для задачи предсказания конкретных товаров из каталога в
   235 тыс. позиций: добавления в корзину совершают около 3 % визитёров, а
   окно оценки — всего 15 дней. Оценка идёт на нескольких сотнях пользователей
   с историей, поэтому метрики шумные; сравнивать модели между собой это не
   мешает, база усреднения у всех одна.

## Этап 4. Ранжирование кандидатов